# Setup environment


</br>


<img src="https://upload.wikimedia.org/wikipedia/commons/e/ec/1000_Genomes_Project.svg" alt="1000 Genomes Project population sampling map" style="max-width: 850px; width: 100%; margin: 1em 0;">

In this exercise we will estimate ancestry proportions from a subset of human 1000 Genomes data. The data set has already been converted to PLINK format and subset to 12 individuals from each of 16 populations. Several populations are expected to be admixed.

The main aim is not just to run ADMIXTURE, but also to see why it is important to check convergence across multiple random seeds.


In [ ]:
# shared tools and data folder
ROOT_PATH=/course/chinacourse2026/shared
TOOL_PATH=${ROOT_PATH}/Software # for standalone tool script including java package
SHARED_PATH=${ROOT_PATH}/ref # For reference database
INPUT_PATH=${ROOT_PATH}/data/admixture_notebook_files  # for input data
INPUT_DATA_DIR=${INPUT_PATH}/input
PRECOMPUTE_DATA_DIR=${INPUT_PATH}/precomputed

mkdir -p ~/sysu2026_day4_pca
cd ~/sysu2026_day4_pca
echo "the working folder is " - `pwd`
cp -sf ${SHARED_PATH}/pcaone_*.json .
cp -sf ${SHARED_PATH}/visFuns.R .
cp -sf ${SHARED_PATH}/newPlotPlink.R .
cp -sf ${SHARED_PATH}/locuszoom_ref/refGeneHG38.gz .
ln -sf ${SHARED_PATH}/locuszoom_ref/hg38 .

In [ ]:
# set up R working space
work_d <- path.expand("~/sysu2026_day4_pca")
setwd(work_d)

geneticMap <- path.expand("~/sysu2026_day4_pca/hg38/genetic_map_GRCh38_chr")
refGenes <- path.expand("~/sysu2026_day4_pca/refGeneHG38.gz")

source("./visFuns.R")
source("./newPlotPlink.R")

In [ ]:
# set up python working space
import os
work_d = os.path.expanduser("~/sysu2026_day4_pca")
os.chdir(work_d)

#  PCA for genotype data using PCAone 


In this exercise we will try to use PCAone to analyse the same data used in the ADMIXTURE exercises. 

Genotypes were called for variable sites with high depth sequencing data from the 1000 genomes project


In [ ]:
# Make new folder and set path to that folder
#make folder 
#rm -rf pca
mkdir -p ~/sysu2026_day4_pca/pca


cp -sf ${PRECOMPUTE_DATA_DIR}/best_seed/human_autosomes_12pp_pcaoneLD02.4.best_seed5.Q ./pca/


##make links to files and add them to the folder
# links to LD-pruned genotype file from admixture analysis
cp -sf ${INPUT_DATA_DIR}/human_autosomes_12pp_pcaoneLD02* ./pca/


echo -e "\n--- files in folder ---"
ls ./pca


echo -e "\n--programs that are installed:--"
which PCAone


Look inside the first lines in the population informaiton file. 

In [ ]:
echo --- first 10 lines of the popluation information ---

head ./pca/human_autosomes_12pp_pcaoneLD02.labels.tsv

echo --- summaries the second column ---
cut -d' ' -f2 ./pca/human_autosomes_12pp_pcaoneLD02.labels.tsv | sort |  uniq -c

echo --- summaries the third column ---
cut -d' ' -f3 ./pca/human_autosomes_12pp_pcaoneLD02.labels.tsv | sort |  uniq -c

In [ ]:
from jupyterquiz import display_quiz
display_quiz('pcaone_quiz1.json')


 
 ## Run PCAone to perform PCA
 First let's get a list of the options in PCAone


In [ ]:
${TOOL_PATH}/PCAone

 
 PCAone is a fast and scalable tool for PCA analysis.
 
 For this small dataset, it will be done in **~5 seconds**

In [ ]:
${TOOL_PATH}/PCAone -b ./pca/human_autosomes_12pp_pcaoneLD02 -k 10 -d 0 -o ./pca/PCAONE_K10 -n 6

Let's look at the output of PCAone. Can you figure out what's stored in these output files?

In [ ]:
ls pca/PCAONE_K10.*

In [ ]:
from jupyterquiz import display_quiz
display_quiz('pcaone_quiz2.json')


Now let's make PCA plots!

In [ ]:
# Read eigenvector matrix calculated by PCAone
e <- read.table("./pca/PCAONE_K10.eigvecs")
# Read population and super-population labels for each individuals
labels<-read.table("./pca/human_autosomes_12pp_pcaoneLD02.labels.tsv",stringsAsFactors=T,head=F)
# use pch for different pop, use color for super-pops
pchs <- 1:20
plot(e[,1:2], pch=as.integer(factor(labels[,2])), col=factor(labels[,3]), xlab="PC1", ylab="PC2", cex=2)
legend("topleft",fill=1:4,levels(labels[,3]))
legend("top",legend=levels(labels[,2]),pch=pchs[seq_along(levels(labels[,2]))],col="black",bty="n",cex=1.2)

Compare with the estimate admixture proportions from ADMIXTURE analysis this morning



In [ ]:
#read in code to plot admixture proportions ( plotAdmix function)
#source("https://raw.githubusercontent.com/GenisGE/evalAdmix/master/visFuns.R")

options(repr.plot.width=12, repr.plot.height=12)
layout(matrix(c(1,1,2,3),nrow=2,by=T),height=c(2,4),width=2:1)
# Read population and super-population labels for each individuals
labels<-read.table("./pca/human_autosomes_12pp_pcaoneLD02.labels.tsv",stringsAsFactors=T,head=F)
pop <- labels[,2]
pop_order <- c("GWD", "MSL", "YRI", "ESN", "ACB", "ASW",
               "GBR", "CEU", "IBS", "TSI",
               "CHB", "CHS", "JPT",
               "MXL", "CLM", "PEL")
ord <- unlist(lapply(pop_order, function(p) which(pop == p)))

admix_cols <- c(AMR="#FF7F00", AFR="#4DAF4A", EUR="#377EB8", EAS="#984EA3")

reorder_components <- function(q) {
  ref <- c("PEL", "YRI", "GBR", "CHB")
  kord <- sapply(ref, function(p) which.max(colMeans(q[pop == p,,drop=FALSE])))
  if(length(unique(kord)) == ncol(q)) q[,kord] else q
}

plot_geo_admix <- function(q, title) {
  plotAdmix(q, pop=pop, ord=ord, rotatelab=45, padj=0.12,
            cex.lab=1.0, cex.main=1.3, main=title,
            colorpal=unname(admix_cols[c("AMR", "AFR", "EUR", "EAS")]))
}
                 
# Read in inferred admixture proportions
q_best <- read.table("./pca/human_autosomes_12pp_pcaoneLD02.4.best_seed5.Q")
q_best <- reorder_components(as.matrix(q_best))
plot_geo_admix(q_best, "ADMIXTURE proportions, K = 4, best seed = 5")
# Read eigenvector matrix calculated by PCAone
e <- read.table("./pca/PCAONE_K10.eigvecs")

pchs <- 1:20
# use pch for different pop, use color for super-pops
plot(e[,1:2], pch=pchs[factor(labels[,2])], col=factor(labels[,3]), xlab="PC1", ylab="PC2", cex=2)
legend("topleft",fill=1:4,levels(labels[,3]))
plot.new()
par(mar=c(0,0,0,0))
legend("top",legend=levels(labels[,2]),pch=pchs[seq_along(levels(labels[,2]))],col="black",bty="n",cex=1.5)

 **Questions**
 - What information do you get from the PCA that you don't get from the ADMIXTURE results?
 - Can you identify the admixed individuals?
 
 Lets see what the other PCs show. 
 

In [ ]:
par(mfrow=c(3,2))
for(pc in 1:5)
  plot(e[,1:2+2*(pc-1)], pch=pchs[as.integer(factor(labels[,2]))], col=factor(labels[,3]), ylab=paste("PC",pc*2),xlab=paste("PC",pc*2-1), cex=2);

 **Questions**
 - How many PCs are used to separate the populations?
 - What do you think is captured on PC 5 and 6?
 - What do you think is captures on PC 9 and 10? 

# PC-based selection

For very recent selection we can look within closely related individuals for example with in Europeans

**Data:**

 - Genotype likelihoods in Beagle format
 - ~150k random SNPs with maf > 5%
 - Four EU populations with ~100 individuals in each
 - whole genome sequencing
 - depth 2-9X (1000 genome project)

 ```
CEU | Europeans in Utah (British)
GBR | Great Britain
IBS | Iberian/Spain
TSI | Italien
```

First let's set the paths


In [ ]:

##make links to files and add them to the folder
# links to genotype likelihood file ( from admixture analysis )
cp -sf ${ROOT_PATH}/data/pca/eu1000g.small.beagle.gz ./pca/

# link to population information file
cp -sf ${ROOT_PATH}/data/pca/eu1000g.sample.Info ./pca/

echo -e "\n--- eu1000* files in folder ---"
ls ./pca/eu1000*


### Explore the input data. 

Take a quick look at the sample data.

First try to get an overview of the dataset by looking at the information file and making a summary using the following code:
 

In [ ]:
# View first lines of sample info file
echo ---- First lines in sample info file ----
head ./pca/eu1000g.sample.Info

echo ---- Count the number of samples from each population ----
cut -f 2 -d " " ./pca/eu1000g.sample.Info | sed 1d| sort | uniq -c


#### How many samples from each country?

Now let's have a look at the genotype likelihood (GL) file that you have created with ANGSD. It is a "beagle format" file called all.beagle.gz - and will be the input file to PCAngsd. The first line in this file is a header line and after that it contains a line for each locus with GLs. By using the unix command wc we can count the number of lines in the file. Next, to get an idea of what the GL file contains try (from the command line) to print the first 9 columns of the first 7 lines of the file:




In [ ]:
zcat ./pca/eu1000g.small.beagle.gz | head -n 7 | cut -f1-9 | column -t

## Ignore the "Broken pipe"

In general, the first three columns of a beagle file contain marker name and the two alleles, allele1 and allele2, present in the locus (in beagle A=0, C=1, G=2, T=3).

All following columns contain genotype likelihoods (three columns for each individual: first GL for homozygote for allele1, then GL for heterozygote and then GL for homozygote for allele2). Note that the GL values sum to one per site for each individuals. This is just a normalization of the genotype likelihoods in order to avoid underflow problems in the beagle software it does not mean that they are genotype probabilities.

 - Based on this, what is the most likely genotype of Ind0 in the first locus and the locus six?

### Selection analysis

Run PCangsd algorithm to estimate the covariance matrix while jointly estimating the individuals allele frequencies.



In [ ]:
pcangsd -b ./pca/eu1000g.small.beagle.gz -o ./pca/EUsmall -t 5

The program estimates the covariance matrix that can then be used for PCA. Look at the output from the program.

 - The algorithm might only need a low number of PCs to estimate the allele freuqencies. A MAP test can be used to find out the significant PCs.

Now plot the results in R:


In [ ]:
 ## R
options(repr.plot.width=7, repr.plot.height=7)
cov <- as.matrix(read.table("./pca/EUsmall.cov"))
e<-eigen(cov)
ID<-read.table("./pca/eu1000g.sample.Info",head=T,stringsAsFactors=T)
plot(e$vectors[,1:2],col=ID$POP,xlab="PC1",ylab="PC2")
legend("topleft",fill=1:4,levels(ID$POP))


 - Does the plot look like you expected? Which populations are close and distant to each other?

Since the European individuals in 1000G are not simple homogeneous disjoint populations it is hard to use PBS/FST or similar statistics to infer selection based on populating differences ( you will learn about these later). However, PCA offers a good description of the differences between individuals without having the define disjoint groups.

Let's try to infer selection along the genome based on the PCA



In [ ]:
pcangsd -b ./pca/eu1000g.small.beagle.gz -o ./pca/EUsmall --selection -t 5 --sites-save


The analysis takes aboud two minutes. We also need to keep track of whether a SNP is used in the analysis or not, which can be done based on the output. Create a file with the SNP location info that you will need to plot the results (the third column indicate if the site is used=1 or not =0):



In [ ]:
# Create file with position and chromosome
paste <(zcat ./pca/eu1000g.small.beagle.gz| cut -f 1 | sed 's/\_/\t/g' | sed 1d ) ./pca/EUsmall.sites  > ./pca/EUsmall.sites.info

head  ./pca/EUsmall.sites.info 


Next, plot the results of the selection scan



In [ ]:
s = scan("./pca/EUsmall.selection")

# convert test statistic to p-value
pval<-pchisq(s,1,lower=FALSE)

## make QQ plot to QC the test statistics
qqPlot(pval)


The above is a QQ plot of the p-values from the selection scan. If the test statistics is good them most point will follow the red line which only a few (<1%) will deviate.

 - Did the test perform well?
 
Finally, let's plot the results of the scan along the genome:  

In [ ]:


## read positions (hg38)
data<-read.delim("./pca/EUsmall.sites.info",colC=c("factor","integer","integer"),head=F)
names(data)<-c("chr","pos","keep")
data <- subset(data,keep==1)
data$pval <- pval 



## make manhatten plot
options(repr.plot.width = 10, repr.plot.height = 6)
manPlot(data$pval,chr=as.integer(data$chr))




Lest zoom in 

In [ ]:

# select sites to plot, 0.5Mb on either side of SNP
leadSNPposition <- data$pos[which.max(s)]

region <- subset(data,chr=="chr2" & pos >  leadSNPposition - 5e5 &  pos < leadSNPposition + 5e5)


#plot
locusZoomNoLD(region$pval,chr=2,pos=region$pos,main="LocusZoom",build=38, geneticMap=geneticMap, refGenes=refGenes)



See if you can make sense of the top hit. What do you think it the relevant gene in  that locus

# Bonus exercise

## Simple example of PCA and MDS

First let's try to perform PCA and MDS on the small matrix from the slides. The below code will input the genotypes into R. 


In [ ]:
#read in data from slides
G <-matrix(c(1,0,2,0,2,0,2,1,1,1,0,1,0,2,1,2,1,1,1,1,1,0,1,0,2,0,1,1,0,2,1,2,0,1,0),5,by=T,
           dimnames=list(paste0("IND",1:5),paste0("SNP",1:7)))
nInd <- nrow(G)

print(G)

In [ ]:
## run the code to start a quiz
from jupyterquiz import display_quiz
display_quiz('pca_quiz1.json')


## MDS 

Let's try to do MDS. First let's calculate the distance. The simple distance measure as seen in the slides is called a Manhattan distance.


In [ ]:

## continue in R
D<-dist(G,upper=T,diag=T,method="manh")
D



 - How many dimensions are used to represent the distances?

Now let's reduce the number of dimension to 2 using MDS and plot the results:

In [ ]:
k2<-cmdscale(D,k=2)

cat("\n Dimension reduction to two dimensions")
k2
cat("\n original Distance between individuals:")
org <- dist(G,upper=T,diag=T,method="manha")
org 

cat("\n Distance between individuals in from the MDS:")
round(D_k2<- dist(k2,upper=T,diag=T),2)




In [ ]:
#plot the results
 plot(k2,pch=16,cex=3,col=1:5+1,ylab="distance 2th dimension",xlab="distance 1. dimension",main="Multiple dimension scaling (MDS)")
 points(k2,pch=as.character(1:5))



 - Can you find any difference in the pairwise distances from the plot and the original pairwise distances?. 

## PCA
First let's try to perform PCA directy on the normalized genotypes without calculating the covariance matrix

 - Why do we normalize the genotypes?

 

In [ ]:
 #first normalize the data do that the mean and variance is the same for each SNP
normalize <- function(x){
    nInd <- nrow(x)
    avg <- colMeans(x)
    M <- x - rep(colMeans(x),each=nInd)
    M <- M/sqrt(2*rep(avg/2*(1-avg/2),each=nInd))
    M
 }
print(G)
 M <- normalize(G)
print(M)
cat("Dimension of M")
dim(M)

 svd <- svd(M)
 ## print the decomposition for M=SDV
 ## u is the eigenvectors
 ## d is eigen values
 print(svd)


The above is the decomposition of the genotypes into the diagonal matrix (d) with eigenvalues, and the left (u) and right (v) eigenvectors such that
$M=U\Sigma V^T$
where $\Sigma$ has the diagonal values of d. Therefore, we can reconstruct the normalized genotypes from U, d and v:


In [ ]:
##make a diagonal matrix with the eigenvalues
SIGMA <-  diag(svd$d)
print(SIGMA)
## using the matrixes from the decomposition we can undo the transformation of our normalized genotypes
M2 <- svd$u%*%SIGMA%*%t(svd$v)
cat("Original normalized genotypes (M):")
round(M,3)
cat("Reconstructed normalized genotypes(M2):")
round(M2,3)

 - Did the reconstruction of the normalized genotypes work?
 - Would you be able to reconstruct the unnormalized (raw) genotypes?

Now try performing PCA based on the covariance matrix instead. To do so we first calculate the covariance matrix:


In [ ]:
 ## calculate the covariance matrix
C <- M %*% t(M)
 print(C)


The covariance matrix also shows the relationship between each individuals with the most similar individuals having a high positive value while the most distant individuals having a negativ value. However, unlike the euclidian distance the diagonal is not zero but instead is it related to the diversity within each individual.

Now let's try to do PCA on this covariance matrix instead

In [ ]:
 ## then perform the PCA by singular value decomposition
 e <- eigen(C)

 ## print first PC
cat("First pricipal component:")
 print(e$vectors[,1])
 ## print first PC
cat("Eigenvalues:")
 print(round(e$values,4))
 ##plot 2 first PC. for the 5 indiviudals
 plot(e$vectors[,1:2],pch=16,cex=3,col=1:5+1,ylab="2. PC",xlab="1. PC",main="Principle component analysis (PCA)")
 points(e$vectors[,1:2],pch=as.character(1:5))
 




 - Do you get the same results using the covariance matrix as using the normalized genotypes directly?
 - Compare the two plots (MDS vs. PCA). Are the capturing the same thing? 

Bonus information:

Unlike MDS, PCA will not remove information, so you are actually able to reconstruct your covariance matrix from the principal components.

In [ ]:
##continue in R
##make a diagonal matrix with the eigenvalues
SIGMA <- diag(e$value)

## transform the PC back to the original data
## using matrix multiplication V SIGMA Vt
out <- e$vectors %*% SIGMA %*% t(e$vectors)
cat("Reconstructed covariance:")
print(out)
cat("Original covariance:")
print(C)
#close R after you are done

Try to also compare the eigenvalues from the decomposition of the normalized genotypes and from the covariance matrix

In [ ]:
cat("Eigenvalues of the covariance matrix:")
 print(round(e$values,4))

cat("Singular values from the normalized genotypes:")
 print(round(svd$d,4))

 - What is the relationship? (hint: try to square one of them by changing the above code)